In [1]:
!pip install -q ipdb
!pip install -q tqdm
!pip install -q transformers
!pip install -q datasets
!pip install -q wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 84.8 MB/s eta 0:00:00


In [2]:
!nvidia-smi

Fri Jun 12 10:55:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os, requests, zipfile, io

files_url = "https://ideami.com/llm_align"

# Downloading proceeds if we detect that one of the key files to download is not present
if not os.path.exists(f"llm.py"):
    print("Downloading files using Python")
    response = requests.get(files_url)
    zipfile.ZipFile(io.BytesIO(response.content)).extractall(".")
else:
    print("you seem to have already downloaded the files. If you wish to re-download them, delete the llm.py file")

In [4]:
import os
# Pytorch
import torch
# Architecture
import transformers
# Import Llama based model
from llm import Llama, ModelArgs

In [8]:
use_orpo = True  # use aligned checkpoint or not
num_answers = 3
temp = 0.6
topk= 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_path = "tokenizers/tok16384"
model_path = "./models/"

if use_orpo==True:
        model_inf, context= "aligned_model.pt", 1024  # ORPO is trained with context of 1024
        print("Mode::Using Orpo aligned model")
else:
        model_inf, context= "base_model.pt", 512  # The original was trained with context of 512
        print("Mode::Using pretrained model without alignment")

print(f"Using model {model_inf}")

# Load model and extract config
# checkpoint = torch.load(os.path.join(model_path, model_inf), map_location=device)
# config = checkpoint.pop("config")

checkpoint = torch.load(os.path.join(model_path, model_inf), map_location=device, weights_only=False)
config = checkpoint.pop("config")


# temporary fix if the model was trained and saved with torch.compile
# The _orig_mod. prefix in your model's state dictionary keys is related to
# how PyTorch handles compiled models, specifically when using the torch.compile function
# When torch.compile is used, PyTorch might wrap the original model in a way that modifies
# the names of its parameters and buffers. This wrapping can prepend a prefix like _orig_mod.
# We remove those wrappings to make the checkpoint compatible with the non compiled version of the model
new_dict = dict()
for k in checkpoint.keys():
        if k.startswith("_orig_mod."):
            #print("Removing _orig_mod wrapping")
            new_dict[k.replace("_orig_mod.", "")] = checkpoint[k]
        else:
            new_dict[k] = checkpoint[k]

# Setup tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(tokenizer_path)
tokenizer.pad_token = tokenizer.eos_token

model_args = ModelArgs(
        dim=config.hidden_size,
        n_layers=config.num_hidden_layers,
        n_heads=config.num_attention_heads,
        n_kv_heads=config.num_key_value_heads,
        vocab_size=config.vocab_size,
        norm_eps=config.rms_norm_eps,
        rope_theta=config.rope_theta,
        max_seq_len=context,
        dropout=config.attention_dropout,
        hidden_dim=config.intermediate_size,
        attention_bias=config.attention_bias,
        mlp_bias=config.mlp_bias
    )

# Instantiate model, load parms, move to device
model = Llama(model_args)
model.load_state_dict(new_dict)
if device.type == 'cuda':
        model = model.to(torch.bfloat16)
        model = model.to(device)
model.eval()

model_size = sum(t.numel() for t in model.parameters())
print(f"Model size: {model_size/1e6:.2f} M parameters")

# Interactive loop
while True:
         qs = input("Enter text (q to quit) >>> ")
         if qs == "":
             continue
         if qs == 'q':
             break

         # we activate chat template only for ORPO model because it was trained with it
         if use_orpo:
            qs = f"<s> <|user|>\n{qs}</s>\n<s> <|assistant|> "

         x = tokenizer.encode(qs)
         x = torch.tensor(x, dtype=torch.long, device=device)[None, ...]

         for ans in range(num_answers):
            with torch.no_grad():
                y = model.generate(
                    x,
                    max_new_tokens=256,
                    temperature=temp,
                    top_k=topk
                )

            response = tokenizer.decode(y[0].tolist(), skip_special_tokens=True)

            output = model.clean_response(response)

            print("################## \n")
            print(f"### Answer {ans+1}: {output}")


Mode::Using Orpo aligned model
Using model aligned_model.pt
Model size: 138.43 M parameters
Enter text (q to quit) >>> Explain what gravity is in simple terms.
################## 

### Answer 1: 
Explain what gravity is in simple terms. 
  
Gravity is a force that exists between the Earth and the Moon, which is a planet that orbits the Sun. It is defined as the force that is exerted on the Earth's surface by the gravitational pull of the Sun and Moon. This force is responsible for driving the Earth and Moon on their axis and is responsible for their rotation. The gravitational force between earth and the Moon is the same as the gravitational force between the Earth and the Moon. 

Gravity is a force that exists between the Earth and the Moon that is exerted by the Moon and the Earth on their axis. 

Gravity is a force that exists between the Earth and the Moon that is exerted by the Moon and the Earth on their axis. 

Gravity is a force that is exerted by the Moon and the Earth on thei

In [9]:
use_orpo = True  # use aligned checkpoint or not
num_answers = 3
temp = 0.8
topk= 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_path = "tokenizers/tok16384"
model_path = "./models/"

if use_orpo==True:
        model_inf, context= "aligned_model.pt", 1024  # ORPO is trained with context of 1024
        print("Mode::Using Orpo aligned model")
else:
        model_inf, context= "base_model.pt", 512  # The original was trained with context of 512
        print("Mode::Using pretrained model without alignment")

print(f"Using model {model_inf}")

# Load model and extract config
# checkpoint = torch.load(os.path.join(model_path, model_inf), map_location=device)
# config = checkpoint.pop("config")

checkpoint = torch.load(os.path.join(model_path, model_inf), map_location=device, weights_only=False)
config = checkpoint.pop("config")


# temporary fix if the model was trained and saved with torch.compile
# The _orig_mod. prefix in your model's state dictionary keys is related to
# how PyTorch handles compiled models, specifically when using the torch.compile function
# When torch.compile is used, PyTorch might wrap the original model in a way that modifies
# the names of its parameters and buffers. This wrapping can prepend a prefix like _orig_mod.
# We remove those wrappings to make the checkpoint compatible with the non compiled version of the model
new_dict = dict()
for k in checkpoint.keys():
        if k.startswith("_orig_mod."):
            #print("Removing _orig_mod wrapping")
            new_dict[k.replace("_orig_mod.", "")] = checkpoint[k]
        else:
            new_dict[k] = checkpoint[k]

# Setup tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(tokenizer_path)
tokenizer.pad_token = tokenizer.eos_token

model_args = ModelArgs(
        dim=config.hidden_size,
        n_layers=config.num_hidden_layers,
        n_heads=config.num_attention_heads,
        n_kv_heads=config.num_key_value_heads,
        vocab_size=config.vocab_size,
        norm_eps=config.rms_norm_eps,
        rope_theta=config.rope_theta,
        max_seq_len=context,
        dropout=config.attention_dropout,
        hidden_dim=config.intermediate_size,
        attention_bias=config.attention_bias,
        mlp_bias=config.mlp_bias
    )

# Instantiate model, load parms, move to device
model = Llama(model_args)
model.load_state_dict(new_dict)
if device.type == 'cuda':
        model = model.to(torch.bfloat16)
        model = model.to(device)
model.eval()

model_size = sum(t.numel() for t in model.parameters())
print(f"Model size: {model_size/1e6:.2f} M parameters")

# Interactive loop
while True:
         qs = input("Enter text (q to quit) >>> ")
         if qs == "":
             continue
         if qs == 'q':
             break

         # we activate chat template only for ORPO model because it was trained with it
         if use_orpo:
            qs = f"<s> <|user|>\n{qs}</s>\n<s> <|assistant|> "

         x = tokenizer.encode(qs)
         x = torch.tensor(x, dtype=torch.long, device=device)[None, ...]

         for ans in range(num_answers):
            with torch.no_grad():
                y = model.generate(
                    x,
                    max_new_tokens=256,
                    temperature=temp,
                    top_k=topk
                )

            response = tokenizer.decode(y[0].tolist(), skip_special_tokens=True)

            output = model.clean_response(response)

            print("################## \n")
            print(f"### Answer {ans+1}: {output}")


Mode::Using Orpo aligned model
Using model aligned_model.pt
Model size: 138.43 M parameters
Enter text (q to quit) >>> What is a food recipe that uses apples?
################## 

### Answer 1: 
What is a food recipe that uses apples? 
  
 Here's a simple recipe for a delicious apple pie recipe with a few other great elements:

1. Fill a large bowl, and top with a mixture of salt, apple extract, and sugar.
2. Soldering Iron: In a large bowl, combine all the ingredients and you are finished with a delicious apple pie recipe.
3. Serve your chosen fruit with a salt, apple extract, and sugar mixture. Let it rest for a few minutes, or until it starts to brown. The combination can be added as a side dish or added to a main course.
4. Cake: Add 1 teaspoon of apple sugar to 1 cup of warm wine. Let it rest for a few minutes, or until it starts to rise and cools.
5. Cake: Add 1 cup of mixed apples and sprinkle with apple juice until well combined. This will add a rich and nutty flavor.
6. Enjoy: